### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heart_failure_followup_survival",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5Z89R",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/519/heart+failure+clinical+records.zip && unzip heart+failure+clinical+records.zip && rm heart+failure+clinical+records.zip && mkdir -p local-data-warehouse/heart_failure_followup_survival && mv heart_failure_clinical_records_dataset.csv local-data-warehouse/heart_failure_followup_survival/
""",
    # References
    academic_reference_bibtex="""@article{chicco2020machine,
  title={Machine learning can predict survival of patients with heart failure from serum creatinine and ejection fraction alone},
  author={Chicco, Davide and Jurman, Giuseppe},
  journal={BMC medical informatics and decision making},
  volume={20},
  number={1},
  pages={16},
  year={2020},
  publisher={Springer}
}
""",
    academic_reference_bibtex_key="chicco2020machine",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We use the data as is from UCI.

- We keep all features of the dataset. The study that introduced the dataset also curated a subset of features. We leave it to the pipeline to select the relevant features for the task.
- There is a "time" feature in the data which is the follow-up time in days. The authors used this in some part of the experiments and showed it improved the predictive performance. We also keep this feature and follow their preprocessing to transform it into months (from days). Note, this does not make the data temporal, as the split is across unseen patients (not across time) and we only have one entry per patient.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DEATH_EVENT",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="DEATH_EVENT",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "heart_failure_clinical_records_dataset.csv")
print("Loaded data shape:", df.shape)

as_cat_type = ["anaemia", "DEATH_EVENT","diabetes", "high_blood_pressure", "sex", "smoking"]
df[as_cat_type] = df[as_cat_type].astype("category")

# Transform time from days to months
df["month_to_follow_up"] = df["time"] / 30.0
df = df.drop(columns=["time"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()